# Phone sampling-rate feasibility audit
New paired development-data execution. This does not validate clinical events, the app detector, or disease specificity. RevalExo is excluded. See saved protocol and coverage ledger.

In [1]:
from pathlib import Path
import sys,json,hashlib,time
import numpy as np,pandas as pd
ROOT=Path.cwd()
if not (ROOT/'src').exists(): ROOT=ROOT.parent
sys.path.insert(0,str(ROOT))
from src.features.sampling_audit import audit_segment
from src.data.voisard_aligned import load_aligned_lower_back
from src.features.voisard import _events_outside_uturn,_walking_bounds
from src.features.felius import list_trials
OUT=ROOT/'data/processed/phone_sampling_v1'
OUT.mkdir(parents=True,exist_ok=True)
protocol={'rates':[25,50,60],'selection':'one lexicographically first trial per existing eligible participant; all participants, no outcome selection','scope':'development signal fidelity only; not app estimator validation or clinical event validation','voisard_gap_policy':'reject any native packet gap, do not interpolate missing capture','felius_rate':'100 Hz assumed from existing provider pipeline; no independent clock validation','filter':'polyphase anti-aliasing; common 8 Hz Butterworth band for waveform comparison; fixed 0.3-3 Hz magnitude peaks','thresholds':'no clinical acceptance thresholds fitted','excluded':'RevalExo; no new classification or healthy/stroke training'}
(OUT/'protocol.json').write_text(json.dumps(protocol,indent=2))
rows=[]; ledger=[]; paths={p.stem.removesuffix('_meta'):p for p in (ROOT/'data/raw/voisard_2025/data').glob('*/*/*/*/*_meta.json')}
base=pd.read_csv(ROOT/'data/processed/conditional_gait_v1/trial_features.csv').sort_values(['participant','trial']).groupby('participant',as_index=False).first()
for r in base.itertuples():
    p=paths[r.trial];m=json.loads(p.read_text()); identity=dict(dataset='Voisard',participant=r.participant,trial=r.trial,cohort=r.pathology_raw)
    try:
        x,info=load_aligned_lower_back(p.parent,r.trial)
        if info['gap_count']:raise ValueError('native packet gap')
        lp,la=_events_outside_uturn(m['leftGaitEvents'],m['uturnBoundaries']);rp,ra=_events_outside_uturn(m['rightGaitEvents'],m['uturnBoundaries'])
        spans=_walking_bounds(lp+rp,la+ra)
        n=0
        for bout,(a,b) in enumerate(spans):
            if b>len(x) or a<0 or b-a<6*m['freq'] or not np.isfinite(x[a:b]).all():continue
            for hz in protocol['rates']: rows.append(dict(**identity,bout=bout,**audit_segment(x[a:b],m['freq'],hz)))
            n+=1
        ledger.append(dict(**identity,status='included' if n else 'no valid six-second straight bout',source_sha256=hashlib.sha256((p.parent/(r.trial+'_raw_data_LB.txt')).read_bytes()).hexdigest()))
    except Exception as e:ledger.append(dict(**identity,status=str(e)))
felius=list_trials().sort_values(['subject','trial_key']).groupby('subject',as_index=False).first()
for r in felius.itertuples():
    identity=dict(dataset='Felius',participant=r.subject,trial=r.trial_key,cohort=r.label)
    try:
        p=r.paths['lowback'];frame=pd.read_csv(p);x=frame[['ax','ay','az']].to_numpy(float)
        # First 20 seconds only, no reference segmentation available.
        x=x[:2000]
        for hz in protocol['rates']:rows.append(dict(**identity,bout=0,**audit_segment(x,100,hz)))
        ledger.append(dict(**identity,status='included; unsegmented first 20s',source_sha256=hashlib.sha256(p.read_bytes()).hexdigest()))
    except Exception as e:ledger.append(dict(**identity,status=str(e)))
df=pd.DataFrame(rows);df.to_csv(OUT/'segments.csv',index=False);pd.DataFrame(ledger).to_csv(OUT/'coverage.csv',index=False)
metrics=['energy_above_target_nyquist','energy_above_8hz','band8_nrmse','peak_match_f1','peak_shift_ms','count_cadence_delta']
people=df.groupby(['dataset','cohort','participant','target_hz'])[metrics].mean().reset_index()
people.to_csv(OUT/'participants.csv',index=False)
summary=people.groupby(['dataset','target_hz'])[metrics].agg(['median',lambda x:x.quantile(.95)])
summary.to_csv(OUT/'summary.csv')
print('NEW EXECUTION, not replay. Coverage:');print(pd.DataFrame(ledger).groupby(['dataset','status']).size().to_string())
print(summary.to_string())
print('Cohort summaries:');print(people.groupby(['dataset','cohort','target_hz'])[['band8_nrmse','peak_match_f1','count_cadence_delta']].median().to_string())
(OUT/'provenance.json').write_text(json.dumps({'module_sha256':hashlib.sha256((ROOT/'src/features/sampling_audit.py').read_bytes()).hexdigest(),'python':sys.version,'numpy':np.__version__,'pandas':pd.__version__,'participants':len(people[['dataset','participant']].drop_duplicates()),'new_execution':True},indent=2))


NEW EXECUTION, not replay. Coverage:
dataset  status                           
Felius   included; unsegmented first 20s      166
Voisard  included                             161
         native packet gap                     96
         no valid six-second straight bout      2
                  energy_above_target_nyquist            energy_above_8hz            band8_nrmse            peak_match_f1            peak_shift_ms            count_cadence_delta           
                                       median <lambda_0>           median <lambda_0>      median <lambda_0>        median <lambda_0>        median <lambda_0>              median <lambda_0>
dataset target_hz                                                                                                                                                                           
Felius  25                           0.034755   0.105210         0.123847   0.283015    0.088167   0.122513      0.974984        1.0     10.000000  14.43

274